In [ ]:
import torch
from google.colab import drive

drive.mount("/content/drive")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
        "GB"
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
GPU: Tesla T4
GPU memory: 14.6 GB


In [ ]:
!pip -q install -U transformers accelerate librosa soundfile jiwer tqdm
!pip -q install "pandas==2.2.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 91.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch

print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Pandas: 2.2.3
PyTorch: 2.11.0+cu128
CUDA available: True


In [ ]:
import torch
from transformers import AutoProcessor, Wav2Vec2ForCTC

MODEL_ID = "agbalu/Fadhma-300M"

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID)
model = model.to(device)
model.eval()

print("Model loaded.")
print("Parameters:", f"{model.num_parameters():,}")
print("Vocabulary size:", model.config.vocab_size)
print("Sampling rate:", processor.feature_extractor.sampling_rate)

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Model loaded.
Parameters: 315,479,720
Vocabulary size: 40
Sampling rate: 16000


In [ ]:
vocab = processor.tokenizer.get_vocab()

vocab_sorted = dict(
    sorted(vocab.items(), key=lambda item: item[1])
)

print(vocab_sorted)

{'[PAD]': 0, '[UNK]': 1, '|': 2, '-': 3, 'a': 4, 'b': 5, 'c': 6, 'd': 7, 'e': 8, 'f': 9, 'g': 10, 'h': 11, 'i': 12, 'j': 13, 'k': 14, 'l': 15, 'm': 16, 'n': 17, 'o': 18, 'p': 19, 'q': 20, 'r': 21, 's': 22, 't': 23, 'u': 24, 'v': 25, 'w': 26, 'x': 27, 'y': 28, 'z': 29, 'č': 30, 'ǧ': 31, 'ɛ': 32, 'ɣ': 33, 'ḍ': 34, 'ḥ': 35, 'ṛ': 36, 'ṣ': 37, 'ṭ': 38, 'ẓ': 39, '<s>': 40, '</s>': 41}


In [ ]:
from pathlib import Path

AUDIO_DIR = Path("/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/pilot_audio")

wav_files = sorted(AUDIO_DIR.glob("*.wav"))

print("Number of WAV files:", len(wav_files))

for p in wav_files[:20]:
    print(p.name)

Number of WAV files: 30
REC010_SEG0055.wav
REC011_SEG0086.wav
REC013_SEG0113.wav
REC014_SEG0018.wav
REC015_SEG0046.wav
REC015_SEG0129.wav
REC016_SEG0017.wav
REC016_SEG0046.wav
REC016_SEG0051.wav
REC016_SEG0081.wav
REC034_SEG0030.wav
REC034_SEG0071.wav
REC034_SEG0082.wav
REC035_SEG0039.wav
REC035_SEG0097.wav
REC036_SEG0016.wav
REC037_SEG0017.wav
REC037_SEG0056.wav
REC038_SEG0080.wav
REC038_SEG0114.wav


In [ ]:
sample_files = wav_files[:5]

print("Selected files:")
for p in sample_files:
    print(p.name)

Selected files:
REC010_SEG0055.wav
REC011_SEG0086.wav
REC013_SEG0113.wav
REC014_SEG0018.wav
REC015_SEG0046.wav


In [ ]:
import librosa
import torch

audio_path = sample_files[3]

audio, sr = librosa.load(
    audio_path,
    sr=16000,
    mono=True
)

inputs = processor(
    audio,
    sampling_rate=16000,
    return_tensors="pt"
)

input_values = inputs.input_values.to(device)

with torch.no_grad():
    logits = model(input_values).logits

predicted_ids = torch.argmax(logits, dim=-1)

prediction = processor.batch_decode(predicted_ids)[0]

print("FILE:")
print(audio_path.name)

print("\nFADHMA PREDICTION:")
print(prediction)

FILE:
REC014_SEG0018.wav

FADHMA PREDICTION:
aṭas i t-yeqqaneqqam


In [ ]:
results = []

for audio_path in sample_files:
    audio, sr = librosa.load(
        audio_path,
        sr=16000,
        mono=True
    )

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        logits = model(input_values).logits

    predicted_ids = torch.argmax(logits, dim=-1)

    prediction = processor.batch_decode(predicted_ids)[0]

    results.append({
        "file": audio_path.name,
        "fadhma_prediction": prediction
    })

    print("=" * 80)
    print(audio_path.name)
    print(prediction)

REC010_SEG0055.wav
axxama a nidi
REC011_SEG0086.wav
ibeqquyen n tnin t yeqqulen cnimeqqaras neqqar-asen iwdan n reḥe tin lebda ara t-ɛicen xrebḥa atɛǧib-as n rrebḥa q tudeɛt-nsen ira xrebḥa wan
REC013_SEG0113.wav
dtobul
REC014_SEG0018.wav
aṭas i t-yeqqaneqqam
REC015_SEG0046.wav
ccḍeḥ n ɛray-is dr di kada ayet wayr wuqqim mačči dteggant din rebɛad diṭ deg enči yeqqaɛnakru aɛgaz d a wuyaɣel maɛma dɛmadi ccḍeḥ ɛad rux ad ceḍḥit ukinnan uceṭṭḥesfiḍ


In [ ]:
import pandas as pd

df_results = pd.DataFrame(results)

df_results

,file,fadhma_prediction
0,REC010_SEG0055.wav,axxama a nidi
1,REC011_SEG0086.wav,ibeqquyen n tnin t yeqqulen cnimeqqaras neqqar...
2,REC013_SEG0113.wav,dtobul
3,REC014_SEG0018.wav,aṭas i t-yeqqaneqqam
4,REC015_SEG0046.wav,ccḍeḥ n ɛray-is dr di kada ayet wayr wuqqim ma...


In [ ]:
OUTPUT_PATH = "/content/drive/MyDrive/Fadhma_300M_Tarifit_qualitative_probe.csv"

df_results.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print("Saved to:")
print(OUTPUT_PATH)

Saved to:
/content/drive/MyDrive/Fadhma_300M_Tarifit_qualitative_probe.csv


In [ ]:
from pathlib import Path
import pandas as pd

matches = list(
    Path("/content/drive/MyDrive").rglob("data/splits/validation.csv")
)

print("Found:")
for p in matches:
    print(p)

Found:


In [ ]:
from pathlib import Path

VAL_DIR = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/"
    "data/processed/mms_corpus_v1_1/validation"
)

print("Exists:", VAL_DIR.exists())

for p in sorted(VAL_DIR.iterdir()):
    print(p.name)

Exists: True
data-00000-of-00001.arrow
dataset_info.json
state.json


In [ ]:
from datasets import load_from_disk

VAL_DIR = (
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/"
    "data/processed/mms_corpus_v1_1/validation"
)

val_ds = load_from_disk(VAL_DIR)

print(val_ds)
print("\nColumns:", val_ds.column_names)
print("\nFeatures:")
print(val_ds.features)

Dataset({
    features: ['input_values', 'input_length', 'labels'],
    num_rows: 133
})

Columns: ['input_values', 'input_length', 'labels']

Features:
{'input_values': List(Value('float32')), 'input_length': Value('int64'), 'labels': List(Value('int64'))}


In [ ]:
example = val_ds[0]

for key, value in example.items():
    if key == "audio":
        print(
            "audio:",
            {
                "sampling_rate": value.get("sampling_rate"),
                "array_shape": value["array"].shape
                if value.get("array") is not None
                else None,
            },
        )
    else:
        print(f"{key}: {value}")

input_values: [0.0012502333847805858, 9.41519028856419e-05, 9.41519028856419e-05, 0.0012502333847805858, 9.41519028856419e-05, 0.0012502333847805858, 0.0012502333847805858, 0.0012502333847805858, 9.41519028856419e-05, 9.41519028856419e-05, 0.0012502333847805858, 9.41519028856419e-05, 0.0012502333847805858, 0.0012502333847805858, 9.41519028856419e-05, 9.41519028856419e-05, 0.0012502333847805858, 0.0012502333847805858, 0.0012502333847805858, 9.41519028856419e-05, 9.41519028856419e-05, 9.41519028856419e-05, 9.41519028856419e-05, 0.0012502333847805858, 9.41519028856419e-05, 9.41519028856419e-05, 0.0012502333847805858, 0.0012502333847805858, 0.0012502333847805858, 9.41519028856419e-05, 9.41519028856419e-05, 9.41519028856419e-05, 9.41519028856419e-05, 9.41519028856419e-05, 9.41519028856419e-05, 0.0012502333847805858, 9.41519028856419e-05, 9.41519028856419e-05, 0.0012502333847805858, 0.0012502333847805858, 9.41519028856419e-05, 9.41519028856419e-05, 9.41519028856419e-05, 0.0012502333847805858

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

for name in [
    "vocab.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
]:
    print(f"\n--- {name} ---")
    matches = list(PROJECT_ROOT.rglob(name))
    for p in matches:
        print(p)


--- vocab.json ---
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/xlsr_tokenizer/vocab.json
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/xlsr_tokenizer_v1_1/vocab.json
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/mms_tokenizer_v1_1/vocab.json
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/mms_M2_tokenizer_v1_1/vocab.json

--- tokenizer_config.json ---

--- special_tokens_map.json ---


In [ ]:
print("\nPossible transcript manifests:\n")

for pattern in ["*.csv", "*.jsonl", "*.json", "*.tsv"]:
    for p in PROJECT_ROOT.rglob(pattern):
        n = p.name.lower()

        if any(k in n for k in [
            "validation",
            "manifest",
            "segment",
            "metadata",
            "corpus"
        ]):
            print(p)


Possible transcript manifests:

/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/.venv_figures/lib/python3.12/site-packages/numpy/core/tests/data/umath-validation-set-arcsinh.csv
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/.venv_figures/lib/python3.12/site-packages/numpy/core/tests/data/umath-validation-set-log2.csv
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/.venv_figures/lib/python3.12/site-packages/numpy/core/tests/data/umath-validation-set-arctanh.csv
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/.venv_figures/lib/python3.12/site-packages/numpy/core/tests/data/umath-validation-set-sin.csv
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/.venv_figures/lib/python3.12/site-packages/numpy/core/tests/data/umath-validation-set-cos.csv
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/.venv_figures/lib/python3.12/site-packages/numpy/core/tests/data/umath-validation-set-cbrt.csv
/content/drive/Othercompute

In [ ]:
example = val_ds[0]

print("input_length:", example["input_length"])
print("number of input samples:", len(example["input_values"]))
print("label IDs:", example["labels"])
print("number of label IDs:", len(example["labels"]))
print(
    "duration approximately:",
    round(example["input_length"] / 16000, 2),
    "seconds"
)

input_length: 52480
number of input samples: 52480
label IDs: [18, 18, 1, 12, 1, 13, 20, 26, 12, 9, 11, 20, 13, 0, 14, 5, 3, 3, 0, 13, 5, 17, 23, 5, 13]
number of label IDs: 25
duration approximately: 3.28 seconds


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

print("VOCAB FILES:")
for p in PROJECT_ROOT.rglob("vocab.json"):
    print(p)

print("\nTOKENIZER CONFIGS:")
for p in PROJECT_ROOT.rglob("tokenizer_config.json"):
    print(p)

VOCAB FILES:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/xlsr_tokenizer/vocab.json
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/xlsr_tokenizer_v1_1/vocab.json
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/mms_tokenizer_v1_1/vocab.json
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/mms_M2_tokenizer_v1_1/vocab.json

TOKENIZER CONFIGS:


In [ ]:
import json

VOCAB_PATH = (
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/"
    "data/processed/mms_tokenizer_v1_1/vocab.json"
)

with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    mms_vocab = json.load(f)

print(mms_vocab)

{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'p': 15, 'q': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'w': 21, 'x': 22, 'y': 23, 'z': 24, 'ǧ': 25, 'ɛ': 26, 'ɣ': 27, 'ʷ': 28, 'ḍ': 29, 'ḥ': 30, 'ṣ': 31, 'ṭ': 32, 'ẓ': 33, '|': 0, '[UNK]': 34, '[PAD]': 35}


In [ ]:
from transformers import Wav2Vec2CTCTokenizer

ref_tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=VOCAB_PATH,
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

print("Vocabulary size:", len(ref_tokenizer))

Vocabulary size: 38


In [ ]:
label_ids = val_ds[0]["labels"]

reference = ref_tokenizer.decode(
    label_ids,
    group_tokens=False
)

print("Label IDs:")
print(label_ids)

print("\nDecoded reference:")
print(reference)

Label IDs:
[18, 18, 1, 12, 1, 13, 20, 26, 12, 9, 11, 20, 13, 0, 14, 5, 3, 3, 0, 13, 5, 17, 23, 5, 13]

Decoded reference:
ssalamuɛlikum necc meryem


In [ ]:
references = [
    ref_tokenizer.decode(
        example["labels"],
        group_tokens=False
    ).strip()
    for example in val_ds
]

print("First 10 references:\n")

for i, ref in enumerate(references[:10]):
    print(f"{i}: {ref}")

First 10 references:

0: ssalamuɛlikum necc meryem
1: aqay ruxxa tnayn uɛecrin sana di hulanda
2: mercex ak nmis n jjiran usiɣ d zi lmeɣrib umi ira aqqay di lmeɣrib ira qqaɣas ad raḥaɣ a urupa ad ggex ad ggex maca
3: umi wsiɣd dda ufix manayenni wa ǧi ca min ira ɣari ddhi rɛqel inu
4: a nec mammec ira ǧjix di lmeɣrib waǧi manayenni uffix dda
5: necc di lmeɣrib ira ɣari lḥurriya inu ira ɣari yemma dd baba ɣari suctma lmuhim ɣari kulci ira teɛicex mammec nneɣni lmuhim ɛawed ayi mammec tuɣa tɛiced
6: di lmeɣrib neccin mammec ira niɛicc
7: ag baba dd yemma wa ɣaneɣ ca ɣaneɣ ca n reḥwayej n teggit en ɣaneɣ can n reḥwayej wa ntegg ca maca waǧi iziyyan wa zeyyan ca aṭṭas am mammec ira tiɛiceɣ ag ruxa mayn aks
8: lmuhim wsiɣd
9: nec ira ɛemmas wa wsiɣd ɣar urupa wsiɣd ddi ṭiyara jjix familya inu jjix lɛaila inu umi wsiɣd wfiɣ manyenni waǧi am mammec ira nniɣ di reɛqer inu


In [ ]:
import torch
from tqdm.auto import tqdm

fadhma_predictions = []

model.eval()

for example in tqdm(val_ds, desc="Fadhma zero-shot"):

    # Already-prepared 16 kHz waveform
    audio = torch.tensor(
        example["input_values"],
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    with torch.inference_mode():
        logits = model(audio).logits

    pred_ids = torch.argmax(logits, dim=-1)

    prediction = processor.batch_decode(pred_ids)[0]

    fadhma_predictions.append(prediction)

print("Done.")
print("Predictions:", len(fadhma_predictions))

Fadhma zero-shot:   0%|          | 0/133 [00:00<?, ?it/s]

Done.
Predictions: 133


In [ ]:
for i in range(10):
    print("=" * 80)
    print(f"SEGMENT {i}")
    print("REFERENCE:", references[i])
    print("FADHMA:   ", fadhma_predictions[i])

SEGMENT 0
REFERENCE: ssalamuɛlikum necc meryem
FADHMA:    a saram u ɛlicom n ccmmerya-m
SEGMENT 1
REFERENCE: aqay ruxxa tnayn uɛecrin sana di hulanda
FADHMA:    a aqay r uxxan tnayen u ɛecrin sɛn adi hulanda
SEGMENT 2
REFERENCE: mercex ak nmis n jjiran usiɣ d zi lmeɣrib umi ira aqqay di lmeɣrib ira qqaɣas ad raḥaɣ a urupa ad ggex ad ggex maca
FADHMA:    mercex aga mis n jiran usiɣezzil meɣrib umi iraq aydi lmeɣrib ilaqaɣ-ac addraḥa ilaqaɣ-as ad raḥeɣ urubadegga reggex maca
SEGMENT 3
REFERENCE: umi wsiɣd dda ufix manayenni wa ǧi ca min ira ɣari ddhi rɛqel inu
FADHMA:    umi wsiɣ dda ufixh ufix man ayenni waǧǧic-a mi riraɣridi di leɛqelinu
SEGMENT 4
REFERENCE: a nec mammec ira ǧjix di lmeɣrib waǧi manayenni uffix dda
FADHMA:    anec ma mma mecra ǧǧix-tm ɣrib wuǧǧi ma  ayen yufa ixeddeɣ
SEGMENT 5
REFERENCE: necc di lmeɣrib ira ɣari lḥurriya inu ira ɣari yemma dd baba ɣari suctma lmuhim ɣari kulci ira teɛicex mammec nneɣni lmuhim ɛawed ayi mammec tuɣa tɛiced
FADHMA:    necdilmeɣribila ɣer 

In [ ]:
import re
import unicodedata

def eval_normalize(text):
    text = unicodedata.normalize("NFC", str(text))
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

refs_eval = [eval_normalize(x) for x in references]
preds_eval = [eval_normalize(x) for x in fadhma_predictions]

In [ ]:
from jiwer import wer, cer

fadhma_wer = wer(refs_eval, preds_eval)
fadhma_cer = cer(refs_eval, preds_eval)

print("=" * 55)
print("FADHMA-300M ZERO-SHOT — TARIFIT VALIDATION")
print("=" * 55)
print(f"Segments : {len(refs_eval)}")
print(f"WER      : {fadhma_wer * 100:.2f}%")
print(f"CER      : {fadhma_cer * 100:.2f}%")

FADHMA-300M ZERO-SHOT — TARIFIT VALIDATION
Segments : 133
WER      : 97.41%
CER      : 55.78%


In [ ]:
from jiwer import wer, cer

per_segment_wer = []
per_segment_cer = []

for ref, pred in zip(refs_eval, preds_eval):
    per_segment_wer.append(wer(ref, pred))
    per_segment_cer.append(cer(ref, pred))

In [ ]:
import pandas as pd
from pathlib import Path

results_df = pd.DataFrame({
    "validation_index": range(len(refs_eval)),
    "duration_seconds": [
        x["input_length"] / 16000
        for x in val_ds
    ],
    "reference": references,
    "fadhma_prediction_raw": fadhma_predictions,
    "reference_eval": refs_eval,
    "prediction_eval": preds_eval,
    "wer": per_segment_wer,
    "cer": per_segment_cer
})

OUTPUT_DIR = Path(
    "/content/drive/Othercomputers/My MacBook Pro/"
    "tarifit_asr_tfm/results/fadhma_zero_shot"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "fadhma_300m_validation_results.csv"

results_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8"
)

print("Saved:", OUTPUT_FILE)

Saved: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/fadhma_zero_shot/fadhma_300m_validation_results.csv


In [ ]:
summary_df = pd.DataFrame([{
    "model": "agbalu/Fadhma-300M",
    "evaluation_split": "validation",
    "num_segments": len(refs_eval),
    "decoding": "greedy",
    "language_model": False,
    "orthographic_mapping": False,
    "WER_percent": fadhma_wer * 100,
    "CER_percent": fadhma_cer * 100
}])

summary_file = OUTPUT_DIR / "fadhma_300m_validation_summary.csv"
summary_df.to_csv(summary_file, index=False)

display(summary_df)

When evaluated zero-shot on the Tarifit validation set using greedy CTC decoding and without language-model rescoring or orthographic conversion, Fadhma-300M obtained a WER of 97.41% and a CER of 55.78%. Although word-level accuracy was very low, the considerably lower CER suggests partial phonetic and graphemic transfer from Kabyle to Tarifit. The result also reflects the mismatch between the Kabyle output vocabulary and the Tarifit transcription convention.